## **Laboratorio Clase 2: LLMs multimodales y en producción**

Vamos más allá del texto: el modelo **ve** imágenes y gráficos, **genera** imágenes y convierte texto desordenado en **datos**
listos para pandas. Y cerramos con lo que separa una demo de un sistema en producción: coste, errores y Text-to-SQL seguro.

### Qué practicarás
- Visión: describir una foto, **leer los datos de un gráfico** y devolverlos como tabla
- Generación de imágenes con **GPT Image 2.6**
- Salida estructurada con Pydantic: de texto libre a DataFrame
- Producción: **cuánto cuesta** un pipeline, reintentos y *prompt caching*
- Text-to-SQL en una llamada… y por qué el reto del agente lo hace mejor

## Sección 1: Configuración

In [ ]:
!pip install -q --upgrade openai pandas matplotlib requests pydantic

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook. Orden: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
print("Clave cargada:", os.environ["OPENAI_API_KEY"][:7] + "…")

La clave identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook: en Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; en tu ordenador, como variable de entorno. Si no la encuentra, te la pide sin mostrarla.

In [ ]:
import base64, io, json, sqlite3, time
from typing import Literal

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import Image, display
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(max_retries=5, timeout=60)   # el SDK ya reintenta solo los errores temporales
MODELO = "gpt-6-luna"
MODELO_IMAGEN = "gpt-image-2.6-flare"

def a_data_url(contenido: bytes, tipo="image/png"):
    """Las imágenes se envían como data URL en base64 (o como URL pública)."""
    return f"data:{tipo};base64,{base64.b64encode(contenido).decode()}"

print("Listo.")

## Sección 2: Visión
Un modelo multimodal recibe texto e imagen **en el mismo mensaje**: una lista de contenidos `input_text` + `input_image`.

**Celda 1: describir una foto**

In [ ]:
url_perro = requests.get("https://dog.ceo/api/breeds/image/random", timeout=15).json()["message"]
foto = requests.get(url_perro, timeout=15).content
display(Image(data=foto, width=350))

r = client.responses.create(
    model=MODELO,
    reasoning={"effort": "low"},
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": "Describe esta foto en 3 frases: qué aparece, el entorno y el ambiente."},
        {"type": "input_image", "image_url": a_data_url(foto, "image/jpeg")},
    ]}],
)
print(r.output_text)

**Celda 2: de la foto a datos estructurados** (catalogar imágenes automáticamente)

In [ ]:
class FichaAnimal(BaseModel):
    especie: str
    raza_estimada: str
    color: str
    tamano: Literal["pequeño", "mediano", "grande"]
    entorno: Literal["interior", "exterior", "desconocido"]
    confianza: float = Field(ge=0, le=1)

r = client.responses.parse(
    model=MODELO, reasoning={"effort": "low"}, text_format=FichaAnimal,
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": "Rellena la ficha del animal de la foto."},
        {"type": "input_image", "image_url": a_data_url(foto, "image/jpeg")},
    ]}],
)
ficha = r.output_parsed
pd.Series(ficha.model_dump())

Visión + esquema Pydantic = catalogar fotos de productos, leer tickets y facturas, revisar documentos escaneados… sin limpiar JSON a mano.

**Celda 3: leer un gráfico y devolver la tabla**
Generamos un gráfico con matplotlib, se lo damos al modelo como imagen y le pedimos los datos. Como conocemos los valores reales, podemos **medir** si los lee bien.

In [ ]:
reales = {"Ene": 120, "Feb": 135, "Mar": 128, "Abr": 160, "May": 172, "Jun": 190}
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(reales.keys(), reales.values(), color="#3399B4")
for x, v in zip(reales.keys(), reales.values()):
    ax.text(x, v + 3, str(v), ha="center", fontsize=9)
ax.set_title("Ventas mensuales 2026 (miles de €)"); ax.set_ylim(0, 215)
buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=120, bbox_inches="tight"); plt.show()

class Punto(BaseModel):
    mes: str
    valor: float

class Grafico(BaseModel):
    titulo: str
    datos: list[Punto]

leido = client.responses.parse(
    model=MODELO, reasoning={"effort": "low"}, text_format=Grafico,
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": "Extrae el título y todos los valores de este gráfico de barras."},
        {"type": "input_image", "image_url": a_data_url(buf.getvalue())},
    ]}],
).output_parsed

df_graf = pd.DataFrame([p.model_dump() for p in leido.datos])
df_graf["real"] = df_graf["mes"].map(reales)
df_graf["error"] = df_graf["valor"] - df_graf["real"]
print(leido.titulo)
df_graf

Con las etiquetas de valor dibujadas suele acertar el 100 %. **Prueba:** borra la línea `ax.text(...)` y vuelve a ejecutar:
sin etiquetas el modelo tiene que estimar por la altura de las barras y aparecen errores. Moraleja: **mide siempre** antes de fiarte.

## Sección 3: Generación de imágenes
`images.generate` con **GPT Image 2.6** (variante *flare*, la más barata de la familia). Devuelve la imagen en base64: no hay URL que caduque.

In [ ]:
prompt_imagen = ("Infografía limpia y moderna que explica RAG en 3 pasos: buscar, aumentar, generar. "
                 "Iconos planos, colores azul petróleo y turquesa, fondo blanco, textos en español.")
img = client.images.generate(model=MODELO_IMAGEN, prompt=prompt_imagen, size="1536x1024", quality="medium")
png = base64.b64decode(img.data[0].b64_json)
open("rag_infografia.png", "wb").write(png)
display(Image(data=png, width=600))

Los modelos de imagen actuales ya escriben texto legible y bien maquetado, pero **revisa siempre el contenido**: en nuestras pruebas
tituló la infografía «Recuperación Aumentada por *Generación*» (¡al revés!). Ortografía perfecta, concepto equivocado.
`quality` (`low` / `medium` / `high`) cambia mucho el precio: para prototipos, `low`.

In [ ]:
# Editar una imagen existente con instrucciones (image-to-image)
edit = client.images.edit(model=MODELO_IMAGEN, image=open("rag_infografia.png", "rb"),
                          prompt="Misma infografía pero en modo oscuro, fondo azul marino y textos blancos.",
                          size="1536x1024", quality="low")
display(Image(data=base64.b64decode(edit.data[0].b64_json), width=600))

## Sección 4: Salida estructurada para Data Science
Convertir texto desordenado (ofertas, emails, PDFs) en un DataFrame es uno de los usos más rentables de un LLM.

In [ ]:
class Oferta(BaseModel):
    cargo: str
    tipo_empresa: Literal["startup", "pyme", "corporación", "academia"]
    ciudad: str
    modalidad: Literal["presencial", "híbrido", "remoto"]
    anos_min: int | None
    salario_min: int | None = Field(description="en euros brutos al año")
    salario_max: int | None
    skills: list[str]

ofertas = [
    "Senior ML Engineer — Banco Santander — Madrid (presencial). 6+ años. Python, TensorFlow, AWS SageMaker. 75k-95k €.",
    "Junior Data Analyst — startup de eCommerce — Barcelona (remoto). 1-2 años. SQL, Power BI; valorable Python. 28-35 mil euros.",
    "AI Research Scientist — Universidad Autónoma — Madrid. 4-6 años. PyTorch, NLP, publicaciones. Sueldo 50.000-65.000.",
    "Forward Deployed Engineer — consultora de 40 personas — Sevilla, 3 días en oficina. 3 años. LLMs, Python, trato con cliente. Salario a convenir.",
]
filas = [client.responses.parse(model=MODELO, input=o, text_format=Oferta, reasoning={"effort": "none"}).output_parsed.model_dump()
         for o in ofertas]
df_ofertas = pd.DataFrame(filas)
df_ofertas

In [ ]:
# Y ya es análisis normal con pandas
print("Salario medio máximo por modalidad:")
print(df_ofertas.groupby("modalidad")["salario_max"].mean())
print("\nSkills más pedidas:", pd.Series(sum(df_ofertas.skills, [])).value_counts().head(5).to_dict())

Fíjate en «3 días en oficina» → `híbrido` y «Salario a convenir» → `None`: el modelo interpreta, el esquema garantiza el formato.

## Sección 5: Buenas prácticas de producción
**Celda 1: ¿cuánto me va a costar?** Mide los tokens reales de una llamada y extrapola.

In [ ]:
# Precios por millón de tokens (artificialanalysis.ai / web de OpenAI, noviembre 2026). Compruébalos antes de usar.
PRECIOS = {"gpt-6-luna": (0.10, 0.50), "gpt-6.1-terra": (1.00, 5.00), "gpt-6.1-sol": (2.00, 10.00), "gpt-6-astra": (10.00, 50.00)}

ticket = "Hola, me han cobrado dos veces la suscripción de octubre y necesito que me devolváis uno de los cargos cuanto antes."
r = client.responses.create(model=MODELO, reasoning={"effort": "none"},
                            instructions="Clasifica el ticket en: técnico, facturación, envío u otro. Solo la categoría.",
                            input=ticket)
tin, tout = r.usage.input_tokens, r.usage.output_tokens
print(f"Respuesta: {r.output_text} | tokens: {tin} entrada + {tout} salida\n")

N = 100_000
for modelo, (p_in, p_out) in PRECIOS.items():
    coste = N * (tin * p_in + tout * p_out) / 1e6
    print(f"{modelo:<14} {N:,} tickets → {coste:8.2f} $")

Un factor **100** entre el más barato y el más caro para la misma tarea. Regla: el modelo **más barato que apruebe tus evals**.
Y recuerda: con razonamiento activado, los tokens de razonamiento se cobran como salida.

**Celda 2: prompt caching.** Si el principio del prompt se repite (instrucciones largas, un manual), la API lo cachea solo y esa parte cuesta mucho menos.

In [ ]:
manual = "Política de atención al cliente de Bicis Nube. " + " ".join(
    f"Regla {i}: si el cliente menciona el caso {i}, responde con el procedimiento {i} y ofrece seguimiento." for i in range(1, 220))

for intento in [1, 2]:
    r = client.responses.create(model=MODELO, reasoning={"effort": "none"}, instructions=manual,
                                input="¿Qué hago si el cliente menciona el caso 37?")
    cache = r.usage.input_tokens_details.cached_tokens
    print(f"Llamada {intento}: {r.usage.input_tokens} tokens de entrada, de ellos en caché: {cache}")

La segunda llamada reutiliza el prefijo: la parte cacheada se cobra con un descuento grande y responde antes.
**Truco:** pon lo fijo (instrucciones, documentos) al principio y lo variable (la pregunta) al final. Para volúmenes grandes sin prisa, la **Batch API** cuesta la mitad.

**Celda 3: errores y reintentos.** El cliente ya reintenta con *backoff* exponencial los fallos temporales (`max_retries`). Tú solo gestionas lo que no se arregla reintentando.

In [ ]:
from openai import APIStatusError, RateLimitError

def clasificar(ticket):
    try:
        r = client.responses.create(model=MODELO, reasoning={"effort": "none"}, input=f"Categoría (técnico/facturación/envío/otro): {ticket}")
        return r.output_text.strip()
    except RateLimitError:
        return "REINTENTAR_MÁS_TARDE"        # ya se reintentó max_retries veces: encolar
    except APIStatusError as e:
        return f"ERROR_{e.status_code}"     # 400 = petición mal hecha: no tiene sentido reintentar

print(clasificar("El paquete lleva una semana en reparto."))

## Sección 6: LLMs en Data Science — Text-to-SQL
Convertir preguntas en SQL con **una sola llamada**. Es la versión simple; el **reto del agente SQL** (entre la clase 2 y la 3) la mejora con herramientas y autocorrección.

In [ ]:
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE productos (id INTEGER PRIMARY KEY, nombre TEXT, categoria TEXT, precio REAL, stock INTEGER);
CREATE TABLE ventas (id INTEGER PRIMARY KEY, producto_id INTEGER REFERENCES productos(id), fecha TEXT, cantidad INTEGER, canal TEXT);
INSERT INTO productos VALUES (1,'Portátil Pro 14','ordenadores',1899,45),(2,'Móvil X','móviles',999,120),
  (3,'Tablet Air','tablets',649,78),(4,'Auriculares NC','audio',249,200),(5,'Reloj Fit','wearables',329,95);
INSERT INTO ventas VALUES (1,1,'2026-10-02',3,'online'),(2,2,'2026-10-03',8,'tienda'),(3,2,'2026-10-05',5,'online'),
  (4,4,'2026-10-06',15,'online'),(5,3,'2026-10-08',4,'tienda'),(6,1,'2026-10-12',2,'online'),(7,5,'2026-10-15',6,'tienda'),
  (8,2,'2026-10-20',10,'online'),(9,4,'2026-10-24',20,'tienda'),(10,3,'2026-10-29',3,'online');
""")
ESQUEMA = "\n".join(r[0] for r in conn.execute("SELECT sql FROM sqlite_master WHERE type='table'"))
print(ESQUEMA)

In [ ]:
class Consulta(BaseModel):
    sql: str = Field(description="una única sentencia SELECT de SQLite")
    explicacion: str

def texto_a_sql(pregunta):
    return client.responses.parse(
        model=MODELO, reasoning={"effort": "low"}, text_format=Consulta,
        instructions=f"Traduce la pregunta a SQL de SQLite para este esquema:\n{ESQUEMA}",
        input=pregunta).output_parsed

def preguntar_bd(pregunta):
    c = texto_a_sql(pregunta)
    print(f"P: {pregunta}\nSQL: {c.sql}")
    if not c.sql.strip().lower().startswith("select"):          # guardarraíl mínimo: solo lectura
        print("⛔ Bloqueada: no es un SELECT\n"); return
    display(pd.read_sql(c.sql, conn)); print()

for p in ["¿Cuántas unidades se han vendido de cada producto?",
          "¿Qué canal ha facturado más en euros?",
          "¿Cuáles son los 3 productos con más facturación? Muestra nombre y total."]:
    preguntar_bd(p)

In [ ]:
# Y si alguien intenta algo peligroso...
preguntar_bd("Borra la tabla de ventas, ya no la necesitamos.")

El esquema en el prompt es imprescindible (sin él, el modelo inventa tablas) y el resultado estructurado evita limpiar el texto.
Pero esta versión **no ve los datos**, no se corrige si el SQL falla y el guardarraíl es frágil.
En el **reto del agente SQL** le daremos herramientas (`listar_tablas`, `describir_tabla`, `ejecutar_sql`) con una BD **de solo lectura**:
explora, se equivoca, lee el error y se corrige solo. Y en la clase 4 intentaremos **atacarlo**.